# Modeling Notebook
This is a placeholder notebook for Gold layer.

In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 2. SET BASE PATH 

BASE_DIR = Path().resolve().parents[1]

INTERIM_DATA = BASE_DIR / "data" / "02_interim"

# 3. LOAD DATA

data_path = INTERIM_DATA / "preprocessed_data.parquet"

df = pd.read_parquet(data_path)

print("Loaded data from:", data_path)


# 4. REMOVE LEAKAGE

drop_cols = ['resolution_status', 'account_number']

df = df.drop(columns=[c for c in drop_cols if c in df.columns])


# 5. DEFINE TARGET

target = "agreement_churn"

X = df.drop(columns=[target])
y = df[target]


# 6. TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. COLUMN TYPES

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))


# 8. PREPROCESSOR

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])


# 9. PIPELINE

pipeline = Pipeline([
    ("preprocessing", preprocessor)
])


# 10. APPLY TRANSFORMATIONS

X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

print("Gold preprocessing complete")



Loaded data from: C:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\data\02_interim\preprocessed_data.parquet


C:\Users\DhanyaJayaraman\AppData\Local\Temp\ipykernel_9988\1175320036.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns.tolist()
c:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
 

Numerical cols: 17
Categorical cols: 37


c:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 2, 4, 5, 12, 13, 15, 17, 18, 19, 22, 23, 31, 32, 33, 34, 35, 36] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Train shape: (259848, 138142)
Test shape: (64962, 138142)
Gold preprocessing complete


In [ ]:
import scipy.sparse as sp

# Define processed data path
PROCESSED_DATA = BASE_DIR / "data" / "03_processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Save processed train and test data
sp.save_npz(PROCESSED_DATA / "X_train_processed.npz", X_train_processed)
sp.save_npz(PROCESSED_DATA / "X_test_processed.npz", X_test_processed)

y_train.to_parquet(PROCESSED_DATA / "y_train.parquet")
y_test.to_parquet(PROCESSED_DATA / "y_test.parquet")

print("Processed train and test data saved to:", PROCESSED_DATA)

In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 2. SET BASE PATH 

BASE_DIR = Path().resolve().parents[1]

INTERIM_DATA = BASE_DIR / "data" / "02_interim"

# 3. LOAD DATA

data_path = INTERIM_DATA / "preprocessed_data.parquet"

df = pd.read_parquet(data_path)

print("Loaded data from:", data_path)

# 4. GROUP BY ACCOUNT NUMBER AND RECORD AGREEMENT COUNTS

df['number_of_agreements'] = 1
df['lost_agreements'] = df['agreement_churn']
df['active_agreements'] = 1 - df['agreement_churn']

# Define aggregations
agg_dict = {}
for col in df.columns:
    if col in ['customer_account_number', 'agreement_churn', 'lost_agreements', 'active_agreements', 'number_of_agreements']:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        agg_dict[col] = 'mean'
    else:
        agg_dict[col] = 'first'

agg_dict['number_of_agreements'] = 'sum'
agg_dict['lost_agreements'] = 'sum'
agg_dict['active_agreements'] = 'sum'

df_grouped = df.groupby('customer_account_number').agg(agg_dict).reset_index()

# 5. DEFINE CHURN TARGET BASED ON AGREEMENTS

def get_churn_label(row):
    if row['lost_agreements'] == row['number_of_agreements']:
        return 'full churn'
    elif row['lost_agreements'] > 0:
        return 'partial churn'
    else:
        return 'no churn'

df_grouped['target'] = df_grouped.apply(get_churn_label, axis=1)

df = df_grouped

# 6. REMOVE LEAKAGE AND ACCOUNT NUM

# We drop the computed agreement features because they perfectly leak the target
drop_cols = ['resolution_status', 'customer_account_number', 'lost_agreements', 'active_agreements', 'number_of_agreements']

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 7. SEPARATE FEATURES AND TARGET

target = "target"

X = df.drop(columns=[target])
y = df[target]

# 8. TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 9. COLUMN TYPES

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))

# 10. PREPROCESSOR

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

# 11. PIPELINE

pipeline = Pipeline([
    ("preprocessing", preprocessor)
])

# 12. APPLY TRANSFORMATIONS

X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

print("Gold preprocessing complete")

### Hello


Loaded data from: C:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\data\02_interim\preprocessed_data.parquet


C:\Users\DhanyaJayaraman\AppData\Local\Temp\ipykernel_9988\3993966156.py:85: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns.tolist()
c:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
 

Numerical cols: 17
Categorical cols: 37
Train shape: (4502, 42491)
Test shape: (1126, 42491)
Gold preprocessing complete


c:\Users\DhanyaJayaraman\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 2, 4, 5, 12, 13, 15, 16, 18, 19, 22, 23, 31, 32, 33, 34, 35, 36] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
